In [7]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


def extract_color_stats(image_bgr):
    """
    Extract mean and standard deviation across BGR and HSV channels.
    """
    # BGR statistics
    b_mean, g_mean, r_mean = np.mean(image_bgr, axis=(0, 1))
    b_std, g_std, r_std = np.std(image_bgr, axis=(0, 1))
    
    # HSV statistics
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h_mean, s_mean, v_mean = np.mean(image_hsv, axis=(0, 1))
    h_std, s_std, v_std = np.std(image_hsv, axis=(0, 1))
    
    return np.array([
        b_mean, g_mean, r_mean, b_std, g_std, r_std,
        h_mean, s_mean, v_mean, h_std, s_std, v_std
    ])


def extract_color_histogram(image_bgr, bins_per_channel=8):
    """
    Extract normalized 3D color histogram across BGR channels.
    """
    hist = cv2.calcHist(
        [image_bgr], [0, 1, 2], None, 
        [bins_per_channel, bins_per_channel, bins_per_channel], 
        [0, 256, 0, 256, 0, 256]
    )
    cv2.normalize(hist, hist)
    return hist.flatten()


def extract_edge_texture_features(image_bgr):
    """
    Extract edge magnitude stats and edge density using Sobel filters.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    
    # Compute Sobel gradients Gx and Gy
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    
    # Compute gradient magnitude
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    
    mean_mag = np.mean(magnitude)
    std_mag = np.std(magnitude)
    
    # Edge density: proportion of pixels above gradient threshold
    edge_threshold = 50.0
    edge_density = np.mean(magnitude > edge_threshold)
    
    return np.array([mean_mag, std_mag, edge_density])


def extract_combined_features(image_path, target_size=(224, 224)):
    """
    Load image and combine color stats, histograms, and edge/texture features.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image at path: {image_path}")
    
    img = cv2.resize(img, target_size)
    
    color_stats = extract_color_stats(img)
    color_hist = extract_color_histogram(img, bins_per_channel=8)
    edge_feats = extract_edge_texture_features(img)
    
    # Return features both as dictionary and combined vector
    return {
        'color_stats': color_stats,
        'histogram': color_hist,
        'edge': edge_feats,
        'combined': np.hstack([color_stats, color_hist, edge_feats])
    }


In [8]:
def load_dataset_and_extract_features(df, image_dir):
    """
    df must contain columns: 'image_id', 'lesion_id', 'label'
    """
    feature_dict = {'color_stats': [], 'histogram': [], 'edge': [], 'combined': []}
    valid_indices = []
    labels = []
    groups = []
    
    for idx, row in df.iterrows():
        img_path = os.path.join(image_dir, f"{row['image_id']}.jpg")
        if not os.path.exists(img_path):
            continue
        
        feats = extract_combined_features(img_path)
        for k in feature_dict.keys():
            feature_dict[k].append(feats[k])
            
        labels.append(row['label'])
        groups.append(row['lesion_id'])
        valid_indices.append(idx)
        
    for k in feature_dict.keys():
        feature_dict[k] = np.array(feature_dict[k])
        
    return feature_dict, np.array(labels), np.array(groups)


def create_lesion_grouped_splits(X, y, groups, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Splits dataset into train, validation, and test sets grouped by lesion_id
    to prevent data leakage across photos of the same lesion.
    """
    val_test_ratio = val_ratio + test_ratio
    gss1 = GroupShuffleSplit(n_splits=1, test_size=val_test_ratio, random_state=seed)
    train_idx, val_test_idx = next(gss1.split(X, y, groups=groups))
    
    relative_test_ratio = test_ratio / val_test_ratio
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_test_ratio, random_state=seed)
    val_rel_idx, test_rel_idx = next(gss2.split(X[val_test_idx], y[val_test_idx], groups=groups[val_test_idx]))
    
    val_idx = val_test_idx[val_rel_idx]
    test_idx = val_test_idx[test_rel_idx]
    
    return train_idx, val_idx, test_idx



In [9]:
def evaluate_models_on_validation(feature_dict, y, train_idx, val_idx):
    """
    Compares models and feature subsets on the Validation Set.
    """
    models = {
        'Majority Class': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    results = []
    
    for feat_name, X in feature_dict.items():
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        for model_name, model in models.items():
            # For tree-based models, scaling is optional, but unscaled/scaled works fine
            if model_name in ['Majority Class', 'Random Forest']:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)
            else:
                model.fit(X_train_scaled, y_train)
                preds = model.predict(X_val_scaled)
                
            acc = accuracy_score(y_val, preds)
            macro_f1 = f1_score(y_val, preds, average='macro')
            
            results.append({
                'Feature Set': feat_name,
                'Model': model_name,
                'Val Accuracy': acc,
                'Val Macro-F1': macro_f1
            })
            
    return pd.DataFrame(results)


def final_test_evaluation(best_model, scaler, X_train, y_train, X_test, y_test, class_names):
    """
    Evaluates the chosen best model ONCE on the unseen Test Set.
    """
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    best_model.fit(X_train_scaled, y_train)
    test_preds = best_model.predict(X_test_scaled)
    
    test_acc = accuracy_score(y_test, test_preds)
    test_macro_f1 = f1_score(y_test, test_preds, average='macro')
    
    print("\n" + "="*50)
    print("FINAL TEST SET EVALUATION (EVALUATED ONLY ONCE)")
    print("="*50)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro-F1: {test_macro_f1:.4f}\n")
    
    print("Detailed Classification Report:")
    print(classification_report(y_test, test_preds, target_names=class_names))
    
    cm = confusion_matrix(y_test, test_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Final Test Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()
    print("Confusion matrix saved to 'confusion_matrix.png'.")


In [10]:
if __name__ == '__main__':
    print("Generating synthetic metadata and sample data for demonstration...")
    np.random.seed(42)
    n_samples = 300
    
    metadata = pd.DataFrame({
        'image_id': [f"ISIC_{i:07d}" for i in range(n_samples)],
        'lesion_id': [f"HAM_{i//2:05d}" for i in range(n_samples)], # 2 images per lesion
        'label': np.random.choice(['Benign', 'Malignant', 'Indeterminate'], size=n_samples, p=[0.6, 0.3, 0.1])
    })
    
    # Generate synthetic features to simulate real extraction
    # (In actual usage, call load_dataset_and_extract_features)
    n_color_stats = 12
    n_hist = 512
    n_edge = 3
    
    synthetic_combined = np.random.randn(n_samples, n_color_stats + n_hist + n_edge)
    feature_dict = {
        'color_stats': synthetic_combined[:, :n_color_stats],
        'histogram': synthetic_combined[:, n_color_stats:n_color_stats+n_hist],
        'edge': synthetic_combined[:, -n_edge:],
        'combined': synthetic_combined
    }
    
    y = metadata['label'].values
    groups = metadata['lesion_id'].values
    class_names = np.unique(y)
    
   train_idx, val_idx, test_idx = create_lesion_grouped_splits(
        feature_dict['combined'], y, groups
    )
    print(f"Data Split Count -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
   val_results = evaluate_models_on_validation(feature_dict, y, train_idx, val_idx)
    print("\nValidation Results Summary:")
    print(val_results.sort_values(by='Val Macro-F1', ascending=False).to_string(index=False))
    
    # Select best combination based on Val Macro-F1
    best_row = val_results.sort_values(by='Val Macro-F1', ascending=False).iloc[0]
    best_feature_set = best_row['Feature Set']
    best_model_name = best_row['Model']
    print(f"\nBest Config selected on Validation: Feature='{best_feature_set}', Model='{best_model_name}'")
    
   X_best = feature_dict[best_feature_set]
    scaler = StandardScaler()
    
    # Select model instance
    best_model = RandomForestClassifier(n_estimators=100, random_state=42)
    if 'Logistic' in best_model_name:
        best_model = LogisticRegression(max_iter=1000, random_state=42)
    elif 'k-NN' in best_model_name:
        best_model = KNeighborsClassifier(n_neighbors=5)
    elif 'Majority' in best_model_name:
        best_model = DummyClassifier(strategy='most_frequent')

    final_test_evaluation(
        best_model=best_model,
        scaler=scaler,
        X_train=X_best[train_idx],
        y_train=y[train_idx],
        X_test=X_best[test_idx],
        y_test=y[test_idx],
        class_names=class_names
    )

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 30)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

possible_paths = [
    Path("data/milk10k"),
    Path("milk10k"),
    Path("../data/milk10k"),
    Path("../../data/milk10k")
]

DATA_DIR = None
for p in possible_paths:
    if (p / "metadata.csv").exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not locate 'metadata.csv' in expected directories.")

IMG_DIR = DATA_DIR / "images"
print(f"Successfully found dataset directory at: {DATA_DIR.resolve()}")

df = pd.read_csv(DATA_DIR / "metadata.csv")
gt = pd.read_csv(DATA_DIR / "supplements" / "training_gt.csv")

print(f"Metadata shape: {df.shape}, Ground Truth shape: {gt.shape}")

target_col = None
for col in ['diagnosis_1', 'dx', 'diagnosis']:
    if col in gt.columns:
        target_col = col
        break

if target_col is None:
    # Fallback: pick any column that is not an ID or image reference
    candidates = [c for c in gt.columns if 'id' not in c.lower() and 'image' not in c.lower()]
    target_col = candidates[0] if len(candidates) > 0 else 'diagnosis_1'

print(f"Using target label column: '{target_col}'")

merged_df = pd.merge(df, gt, on="lesion_id", how="inner")


def extract_color_stats(image_bgr):
    b_mean, g_mean, r_mean = np.mean(image_bgr, axis=(0, 1))
    b_std, g_std, r_std = np.std(image_bgr, axis=(0, 1))
    
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h_mean, s_mean, v_mean = np.mean(image_hsv, axis=(0, 1))
    h_std, s_std, v_std = np.std(image_hsv, axis=(0, 1))
    
    return np.array([
        b_mean, g_mean, r_mean, b_std, g_std, r_std,
        h_mean, s_mean, v_mean, h_std, s_std, v_std
    ])


def extract_color_histogram(image_bgr, bins_per_channel=8):
    hist = cv2.calcHist(
        [image_bgr], [0, 1, 2], None, 
        [bins_per_channel, bins_per_channel, bins_per_channel], 
        [0, 256, 0, 256, 0, 256]
    )
    cv2.normalize(hist, hist)
    return hist.flatten()


def extract_edge_texture_features(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    
    mean_mag = np.mean(magnitude)
    std_mag = np.std(magnitude)
    edge_density = np.mean(magnitude > 50.0)
    
    return np.array([mean_mag, std_mag, edge_density])


def extract_combined_features(image_path, target_size=(224, 224)):
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Could not load image at path: {image_path}")
    
    img = cv2.resize(img, target_size)
    
    color_stats = extract_color_stats(img)
    color_hist = extract_color_histogram(img, bins_per_channel=8)
    edge_feats = extract_edge_texture_features(img)
    
    return {
        'color_stats': color_stats,
        'histogram': color_hist,
        'edge': edge_feats,
        'combined': np.hstack([color_stats, color_hist, edge_feats])
    }

print("Extracting features from MILK10k dataset images...")
feature_dict = {'color_stats': [], 'histogram': [], 'edge': [], 'combined': []}
valid_labels = []
valid_groups = []

subset_df = merged_df.head(2000) 

for _, row in subset_df.iterrows():
    img_id = row['image_id'] if 'image_id' in row else row.get('isic_id')
    img_path = IMG_DIR / f"{img_id}.jpg"
    
    if not IMG_DIR.exists() or not img_path.exists():
        continue
    
    try:
        feats = extract_combined_features(img_path)
        for k in feature_dict.keys():
            feature_dict[k].append(feats[k])
            
        valid_labels.append(row[target_col])
        valid_groups.append(row['lesion_id'])
    except Exception as e:
        continue

for k in feature_dict.keys():
    feature_dict[k] = np.array(feature_dict[k])

y = np.array(valid_labels)
groups = np.array(valid_groups)
class_names = np.unique(y)

print(f"Successfully processed {len(y)} valid images across classes: {class_names}")

def create_lesion_grouped_splits(X, y_labels, lesion_groups, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    val_test_ratio = val_ratio + test_ratio
    gss1 = GroupShuffleSplit(n_splits=1, test_size=val_test_ratio, random_state=seed)
    train_idx, val_test_idx = next(gss1.split(X, y_labels, groups=lesion_groups))
    
    relative_test_ratio = test_ratio / val_test_ratio
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_test_ratio, random_state=seed)
    val_rel_idx, test_rel_idx = next(gss2.split(X[val_test_idx], y_labels[val_test_idx], groups=lesion_groups[val_test_idx]))
    
    val_idx = val_test_idx[val_rel_idx]
    test_idx = val_test_idx[test_rel_idx]
    
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = create_lesion_grouped_splits(feature_dict['combined'], y, groups)
print(f"Data Split Count -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")


def evaluate_models_on_validation(feat_dict, labels, tr_idx, v_idx):
    models = {
        'Majority Class': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    }
    
    results = []
    for feat_name, X in feat_dict.items():
        X_tr, y_tr = X[tr_idx], labels[tr_idx]
        X_v, y_v = X[v_idx], labels[v_idx]
        
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)
        
        for model_name, model in models.items():
            if model_name in ['Majority Class', 'Random Forest']:
                model.fit(X_tr, y_tr)
                preds = model.predict(X_v)
            else:
                model.fit(X_tr_scaled, y_tr)
                preds = model.predict(X_v_scaled)
                
            acc = accuracy_score(y_v, preds)
            macro_f1 = f1_score(y_v, preds, average='macro', zero_division=0)
            
            results.append({
                'Feature Set': feat_name,
                'Model': model_name,
                'Val Accuracy': acc,
                'Val Macro-F1': macro_f1
            })
    return pd.DataFrame(results)

val_results = evaluate_models_on_validation(feature_dict, y, train_idx, val_idx)
print("\nValidation Results Summary:")
print(val_results.sort_values(by='Val Macro-F1', ascending=False).to_string(index=False))

best_row = val_results.sort_values(by='Val Macro-F1', ascending=False).iloc[0]
best_feature_set = best_row['Feature Set']
best_model_name = best_row['Model']
print(f"\nBest Config selected on Validation: Feature='{best_feature_set}', Model='{best_model_name}'")


def final_test_evaluation(best_model, scaler, X_tr, y_tr, X_te, y_te, classes):
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)
    
    best_model.fit(X_tr_scaled, y_tr)
    test_preds = best_model.predict(X_te_scaled)
    
    test_acc = accuracy_score(y_te, test_preds)
    test_macro_f1 = f1_score(y_te, test_preds, average='macro', zero_division=0)
    
    print("\n" + "="*50)
    print("FINAL TEST SET EVALUATION (EVALUATED ONLY ONCE)")
    print("="*50)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro-F1: {test_macro_f1:.4f}\n")
    print("Detailed Classification Report:")
    print(classification_report(y_te, test_preds, target_names=[str(c) for c in classes], zero_division=0))
    
    cm = confusion_matrix(y_te, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[str(c) for c in classes], yticklabels=[str(c) for c in classes])
    plt.title('Final Test Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    os.makedirs('outputs', exist_ok=True)
    plt.savefig('outputs/confusion_matrix.png')
    plt.close()
    print("Confusion matrix successfully saved to 'outputs/confusion_matrix.png'.")

X_best = feature_dict[best_feature_set]
scaler = StandardScaler()

if 'Random Forest' in best_model_name:
    best_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
elif 'Logistic' in best_model_name:
    best_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
elif 'k-NN' in best_model_name:
    best_model = KNeighborsClassifier(n_neighbors=5)
else:
    best_model = DummyClassifier(strategy='most_frequent')

final_test_evaluation(
    best_model=best_model,
    scaler=scaler,
    X_tr=X_best[train_idx],
    y_tr=y[train_idx],
    X_te=X_best[test_idx],
    y_te=y[test_idx],
    classes=class_names
)

Successfully found dataset directory at: /Users/nairacampos/Desktop/MILK10k-Lesion-Classification/MILK10k-Lesion-Classification/data/milk10k
Metadata shape: (10480, 17), Ground Truth shape: (5240, 12)
Using target label column: 'AKIEC'
Extracting features from MILK10k dataset images...
Successfully processed 2000 valid images across classes: [0 1]
Data Split Count -> Train: 1399, Val: 300, Test: 301

Validation Results Summary:
Feature Set               Model  Val Accuracy  Val Macro-F1
   combined       Random Forest      0.930000      0.525281
  histogram       Random Forest      0.926667      0.522569
   combined Logistic Regression      0.736667      0.505993
  histogram Logistic Regression      0.723333      0.497974
color_stats      Majority Class      0.930000      0.481865
  histogram      Majority Class      0.930000      0.481865
       edge      Majority Class      0.930000      0.481865
       edge          k-NN (k=5)      0.930000      0.481865
   combined      Majority Cl

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

possible_paths = [
    Path("data/milk10k"),
    Path("milk10k"),
    Path("../data/milk10k"),
    Path("../../data/milk10k")
]

DATA_DIR = None
for p in possible_paths:
    if (p / "metadata.csv").exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not locate 'metadata.csv' in expected directories ('data/milk10k').")

IMG_DIR = DATA_DIR / "images"
print(f"Successfully found dataset directory at: {DATA_DIR.resolve()}")

df = pd.read_csv(DATA_DIR / "metadata.csv")
gt = pd.read_csv(DATA_DIR / "supplements" / "training_gt.csv")

print(f"Metadata shape: {df.shape}, Ground Truth shape: {gt.shape}")

target_col = None
for col in ['diagnosis', 'diagnostic_category', 'dx', 'diagnosis_1']:
    if col in gt.columns:
        target_col = col
        break

if target_col is None:
    # Fallback to any string/categorical column in gt that isn't an ID
    candidates = [c for c in gt.select_dtypes(include=['object']).columns if 'id' not in c.lower()]
    target_col = candidates[0] if len(candidates) > 0 else gt.columns[1]

print(f"Using target label column: '{target_col}'")

merged_df = pd.merge(df, gt, on="lesion_id", how="inner")


def extract_color_stats(image_bgr):
    """Extract mean and standard deviation across BGR and HSV channels[cite: 1]."""
    b_mean, g_mean, r_mean = np.mean(image_bgr, axis=(0, 1))
    b_std, g_std, r_std = np.std(image_bgr, axis=(0, 1))
    
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    h_mean, s_mean, v_mean = np.mean(image_hsv, axis=(0, 1))
    h_std, s_std, v_std = np.std(image_hsv, axis=(0, 1))
    
    return np.array([
        b_mean, g_mean, r_mean, b_std, g_std, r_std,
        h_mean, s_mean, v_mean, h_std, s_std, v_std
    ])


def extract_color_histogram(image_bgr, bins_per_channel=8):
    """Extract normalized 3D color histogram across BGR channels[cite: 1]."""
    hist = cv2.calcHist(
        [image_bgr], [0, 1, 2], None, 
        [bins_per_channel, bins_per_channel, bins_per_channel], 
        [0, 256, 0, 256, 0, 256]
    )
    cv2.normalize(hist, hist)
    return hist.flatten()


def extract_edge_texture_features(image_bgr):
    """Extract edge magnitude stats and edge density using Sobel filters[cite: 1]."""
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    
    mean_mag = np.mean(magnitude)
    std_mag = np.std(magnitude)
    edge_density = np.mean(magnitude > 50.0)
    
    return np.array([mean_mag, std_mag, edge_density])


def extract_combined_features(image_path, target_size=(224, 224)):
    """Load image and combine color stats, histograms, and edge/texture features[cite: 1]."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Could not load image at path: {image_path}")
    
    img = cv2.resize(img, target_size)
    
    color_stats = extract_color_stats(img)
    color_hist = extract_color_histogram(img, bins_per_channel=8)
    edge_feats = extract_edge_texture_features(img)
    
    return {
        'color_stats': color_stats,
        'histogram': color_hist,
        'edge': edge_feats,
        'combined': np.hstack([color_stats, color_hist, edge_feats])
    }


print("Extracting features from real MILK10k images...")
feature_dict = {'color_stats': [], 'histogram': [], 'edge': [], 'combined': []}
valid_labels = []
valid_groups = []

# Process all merged rows (or use merged_df.head(N) if you want a quicker test run)
for _, row in merged_df.iterrows():
    img_id = row['image_id'] if 'image_id' in row else row.get('isic_id')
    img_path = IMG_DIR / f"{img_id}.jpg"
    
    if not img_path.exists():
        continue
    
    try:
        feats = extract_combined_features(img_path)
        for k in feature_dict.keys():
            feature_dict[k].append(feats[k])
            
        valid_labels.append(row[target_col])
        valid_groups.append(row['lesion_id'])
    except Exception as e:
        continue

for k in feature_dict.keys():
    feature_dict[k] = np.array(feature_dict[k])

y = np.array(valid_labels)
groups = np.array(valid_groups)
class_names = np.unique(y)

print(f"Successfully processed {len(y)} valid images across classes: {list(class_names)}")


def create_lesion_grouped_splits(X, y_labels, lesion_groups, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """Splits dataset into train, val, and test grouped by lesion_id to prevent data leakage[cite: 1, 4]."""
    val_test_ratio = val_ratio + test_ratio
    gss1 = GroupShuffleSplit(n_splits=1, test_size=val_test_ratio, random_state=seed)
    train_idx, val_test_idx = next(gss1.split(X, y_labels, groups=lesion_groups))
    
    relative_test_ratio = test_ratio / val_test_ratio
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_test_ratio, random_state=seed)
    val_rel_idx, test_rel_idx = next(gss2.split(X[val_test_idx], y_labels[val_test_idx], groups=lesion_groups[val_test_idx]))
    
    val_idx = val_test_idx[val_rel_idx]
    test_idx = val_test_idx[test_rel_idx]
    
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = create_lesion_grouped_splits(feature_dict['combined'], y, groups)
print(f"Data Split Count -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

def evaluate_models_on_validation(feat_dict, labels, tr_idx, v_idx):
    models = {
        'Majority Class': DummyClassifier(strategy='most_frequent'),
        'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
        'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    }
    
    results = []
    for feat_name, X in feat_dict.items():
        X_tr, y_tr = X[tr_idx], labels[tr_idx]
        X_v, y_v = X[v_idx], labels[v_idx]
        
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)
        
        for model_name, model in models.items():
            if model_name in ['Majority Class', 'Random Forest']:
                model.fit(X_tr, y_tr)
                preds = model.predict(X_v)
            else:
                model.fit(X_tr_scaled, y_tr)
                preds = model.predict(X_v_scaled)
                
            acc = accuracy_score(y_v, preds)
            macro_f1 = f1_score(y_v, preds, average='macro', zero_division=0)
            
            results.append({
                'Feature Set': feat_name,
                'Model': model_name,
                'Val Accuracy': acc,
                'Val Macro-F1': macro_f1
            })
    return pd.DataFrame(results)

val_results = evaluate_models_on_validation(feature_dict, y, train_idx, val_idx)
print("\nValidation Results Summary:")
print(val_results.sort_values(by='Val Macro-F1', ascending=False).to_string(index=False))

best_row = val_results.sort_values(by='Val Macro-F1', ascending=False).iloc[0]
best_feature_set = best_row['Feature Set']
best_model_name = best_row['Model']
print(f"\nBest Config selected on Validation: Feature='{best_feature_set}', Model='{best_model_name}'")

def final_test_evaluation(best_model, scaler, X_tr, y_tr, X_te, y_te, classes):
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_te_scaled = scaler.transform(X_te)
    
    best_model.fit(X_tr_scaled, y_tr)
    test_preds = best_model.predict(X_te_scaled)
    
    test_acc = accuracy_score(y_te, test_preds)
    test_macro_f1 = f1_score(y_te, test_preds, average='macro', zero_division=0)
    
    print("\n" + "="*50)
    print("FINAL TEST SET EVALUATION (EVALUATED ONLY ONCE)")
    print("="*50)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro-F1: {test_macro_f1:.4f}\n")
    print("Detailed Classification Report:")
    print(classification_report(y_te, test_preds, target_names=[str(c) for c in classes], zero_division=0))
    
    cm = confusion_matrix(y_te, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=[str(c) for c in classes], yticklabels=[str(c) for c in classes])
    plt.title('Final Test Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    os.makedirs('outputs', exist_ok=True)
    plt.savefig('outputs/confusion_matrix.png')
    plt.close()
    print("Confusion matrix successfully saved to 'outputs/confusion_matrix.png'.")

X_best = feature_dict[best_feature_set]
scaler = StandardScaler()

if 'Random Forest' in best_model_name:
    best_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
elif 'Logistic' in best_model_name:
    best_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
elif 'k-NN' in best_model_name:
    best_model = KNeighborsClassifier(n_neighbors=5)
else:
    best_model = DummyClassifier(strategy='most_frequent')

final_test_evaluation(
    best_model=best_model,
    scaler=scaler,
    X_tr=X_best[train_idx],
    y_tr=y[train_idx],
    X_te=X_best[test_idx],
    y_te=y[test_idx],
    classes=class_names
)

Successfully found dataset directory at: /Users/nairacampos/Desktop/MILK10k-Lesion-Classification/MILK10k-Lesion-Classification/data/milk10k
Metadata shape: (10480, 17), Ground Truth shape: (5240, 12)
Using target label column: 'AKIEC'
Extracting features from real MILK10k images...


/var/folders/8p/yrh_lyh11bng196gr45jj1qm0000gn/T/ipykernel_4936/1885121541.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  candidates = [c for c in gt.select_dtypes(include=['object']).columns if 'id' not in c.lower()]


Successfully processed 10480 valid images across classes: [np.int64(0), np.int64(1)]
Data Split Count -> Train: 7336, Val: 1572, Test: 1572

Validation Results Summary:
Feature Set               Model  Val Accuracy  Val Macro-F1
   combined       Random Forest      0.948473      0.510290
       edge       Random Forest      0.926845      0.505773
  histogram       Random Forest      0.947837      0.498506
color_stats       Random Forest      0.937023      0.493634
color_stats      Majority Class      0.947837      0.486610
  histogram      Majority Class      0.947837      0.486610
       edge      Majority Class      0.947837      0.486610
   combined      Majority Class      0.947837      0.486610
       edge          k-NN (k=5)      0.944656      0.485770
  histogram          k-NN (k=5)      0.944020      0.485602
   combined          k-NN (k=5)      0.944020      0.485602
color_stats          k-NN (k=5)      0.943384      0.485434
   combined Logistic Regression      0.667303      